In [1]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

def process_and_resize_split(split_name, csv_path, img_dir, output_dir, target_size=(384, 384), create_mini=False):
    out_img_dir = os.path.join(output_dir, "images_resized")
    os.makedirs(out_img_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    valid_rows = []
    corrupted = 0
    
    print(f"Đang xử lý tập {split_name.upper()} ({len(df)} ảnh)")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_name = row['image_file']
        img_path = os.path.join(img_dir, img_name)
        out_img_path = os.path.join(out_img_dir, img_name)
        
        try:
            if not os.path.exists(out_img_path):
                img = cv2.imread(img_path)
                if img is None:
                    corrupted += 1
                    continue
                
                img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
                cv2.imwrite(out_img_path, img_resized)
            
            valid_rows.append(row)
            
        except Exception:
            corrupted += 1
            continue
            
    cleaned_df = pd.DataFrame(valid_rows)
    out_csv_path = os.path.join(output_dir, f"cleaned_{split_name}.csv")
    cleaned_df.to_csv(out_csv_path, index=False)
    
    print(f"[{split_name.upper()}] Hoàn tất! Đã loại bỏ {corrupted} ảnh lỗi. Data sạch còn: {len(cleaned_df)} mẫu.")
    
    if create_mini:
        mini_df = cleaned_df.sample(frac=0.1, random_state=42) 
        mini_csv_path = os.path.join(output_dir, f"mini_{split_name}.csv")
        mini_df.to_csv(mini_csv_path, index=False)
        print(f"-> [{split_name.upper()}] Đã tạo tập mini dataset: {len(mini_df)} mẫu để tuning.")




LOCAL_DATA_DIR = "./data"  
LOCAL_IMG_DIR = os.path.join(LOCAL_DATA_DIR, "images")

OUTPUT_DIR = "./data_ready_for_kaggle" 
os.makedirs(OUTPUT_DIR, exist_ok=True)

splits = [
    {"name": "train", "csv": os.path.join(LOCAL_DATA_DIR, "processed_train.csv"), "make_mini": True},
    {"name": "dev", "csv": os.path.join(LOCAL_DATA_DIR, "processed_dev.csv"), "make_mini": False},
    {"name": "test", "csv": os.path.join(LOCAL_DATA_DIR, "processed_test.csv"), "make_mini": False}
]

for split in splits:
    if os.path.exists(split["csv"]):
        process_and_resize_split(
            split_name=split["name"],
            csv_path=split["csv"],
            img_dir=LOCAL_IMG_DIR,
            output_dir=OUTPUT_DIR,
            target_size=(384, 384),
            create_mini=split["make_mini"]
        )
    else:
        print(f"\n[Cảnh báo] Không tìm thấy file {split['csv']}")

print(f"XONG! Toàn bộ dữ liệu sẵn sàng đã được lưu tại thư mục: {OUTPUT_DIR}")

Đang xử lý tập TRAIN (41236 ảnh)


100%|██████████| 41236/41236 [02:38<00:00, 260.41it/s]


[TRAIN] Hoàn tất! Đã loại bỏ 0 ảnh lỗi. Data sạch còn: 41236 mẫu.
-> [TRAIN] Đã tạo tập mini dataset: 4124 mẫu để tuning.
Đang xử lý tập DEV (10002 ảnh)


100%|██████████| 10002/10002 [00:36<00:00, 277.16it/s]


[DEV] Hoàn tất! Đã loại bỏ 0 ảnh lỗi. Data sạch còn: 10002 mẫu.
Đang xử lý tập TEST (10001 ảnh)


100%|██████████| 10001/10001 [00:37<00:00, 269.87it/s]


[TEST] Hoàn tất! Đã loại bỏ 0 ảnh lỗi. Data sạch còn: 10001 mẫu.
XONG! Toàn bộ dữ liệu sẵn sàng đã được lưu tại thư mục: ./data_ready_for_kaggle
